<a href="https://colab.research.google.com/github/elhamod/BA305_Fall_2026/blob/main/Session%2019%20-%20MAKEUP%20FRI%20OCT%2030%20-%20FINAL%20PREP/final_prep_himalaya.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Final Practice: Planning a Himalayan Expedition**

You work for an agency that runs guided expeditions in the Nepal Himalaya. Before each season, operations has two planning questions:

1. **How many oxygen sets should we bring for an expedition?** A missing oxygen set high on the mountain costs the agency **10 times** what carrying a spare one does.
2. **Which climbers are at high risk of dying?**

**It is the end of 2014.** You build your models on the seasons 2000–2014, then see how they do on the next five (2015–2019).

**Practice, not graded.** Open this notebook in Colab (button above) and use **Gemini in Colab** for the code: click the Gemini (✦) button at the top right, or "Generate" in an empty code cell. Compare your results with your classmates': this is a friendly competition. Write the one-sentence answers yourself. On the Final you read output, you don't run code.

| Part | Topic | Time |
| --- | --- | --- |
| 1 | Know your data | ~20 min |
| 2 | Modeling and evaluation | ~20 min |
| 3 | Interpretation and drilling deeper | ~20 min |

**Data:** The Himalayan Database via TidyTuesday (2020-09-22): expeditions and climbers on Nepal's peaks, 1905–2019.

## Setup *(just run these cells)*

In [ ]:
# Tools
import pandas as pd

# Load the three Himalayan Database files
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2020/2020-09-22/'
members = pd.read_csv(url + 'members.csv')
exps = pd.read_csv(url + 'expeditions.csv')
peaks = pd.read_csv(url + 'peaks.csv')

In [ ]:
# Expedition table: one row per expedition, ending with the number of oxygen sets its climbers used.

# Per expedition, count from the climber list: team size, hired staff, and climbers who used oxygen.
# (One expedition ID is used twice, so expeditions are matched on ID and year.)
key = ['expedition_id', 'year']
team = members.groupby(key).agg(team_size=('member_id', 'count'),
                                hired_staff=('hired', 'sum'),
                                o2_sets=('oxygen_used', 'sum')).reset_index()

# Height of each peak
peak_height = peaks[['peak_id', 'height_metres']].rename(columns={'height_metres': 'peak_height_metres'})

# Join the expedition details, the peak height and the counts
expeditions = exps[['expedition_id', 'peak_id', 'year', 'season', 'basecamp_date', 'termination_date',
                    'oxygen_used', 'termination_reason']]
expeditions = expeditions.merge(peak_height, on='peak_id', how='left').merge(team, on=key)

# Put the columns in order, with the target last
expeditions = expeditions[['expedition_id', 'peak_id', 'year', 'season', 'basecamp_date', 'termination_date',
                           'peak_height_metres', 'team_size', 'hired_staff', 'oxygen_used', 'termination_reason',
                           'o2_sets']]

# Turn the date text into real dates
expeditions['basecamp_date'] = pd.to_datetime(expeditions['basecamp_date'])
expeditions['termination_date'] = pd.to_datetime(expeditions['termination_date'])
expeditions

In [ ]:
# Climber table: one row per climber, ending with whether they died

# Add the height of the peak each climber attempted
climbers = members.merge(peak_height, on='peak_id', how='left')

# Keep these columns, with the target last
climbers = climbers[['member_id', 'peak_id', 'year', 'season', 'sex', 'age', 'citizenship', 'hired',
                     'peak_height_metres', 'highpoint_metres', 'death_cause', 'died']]
climbers

In [ ]:
# Train on the seasons 2000-2014; test on the next five (2015-2019)
TRAIN_START_YEAR = 2000
TEST_START_YEAR = 2015

### What each column means

**`expeditions`** — one row per expedition

| Column | Meaning |
| --- | --- |
| `expedition_id` | Code for the expedition |
| `peak_id` | Code for the peak (for example `EVER` = Everest) |
| `year`, `season` | When the expedition took place |
| `basecamp_date` | Date the team reached base camp |
| `termination_date` | Date the expedition ended |
| `peak_height_metres` | Height of the peak |
| `team_size` | Number of climbers listed on the expedition, hired staff included |
| `hired_staff` | Number of listed climbers who were hired staff (for example Sherpa guides) |
| `oxygen_used` | Whether anyone on the expedition used bottled oxygen |
| `termination_reason` | Why the expedition ended (reached the summit, bad weather, an accident, ...) |
| `o2_sets` | **Target for Question 3:** number of listed climbers who used bottled oxygen |

**`climbers`** — one row per climber on an expedition

| Column | Meaning |
| --- | --- |
| `member_id` | Code for the climber on that expedition |
| `peak_id`, `year`, `season`, `peak_height_metres` | Same as in `expeditions` |
| `sex`, `age`, `citizenship` | The climber |
| `hired` | Whether the climber was hired staff rather than a paying client |
| `highpoint_metres` | Highest point the climber reached |
| `death_cause` | Cause of death, for climbers who died |
| `died` | **Target for Question 4:** whether the climber died on the expedition |

# **Part 1: Know your data** (~20 min)

Missing values, duplicates and distinct values, already counted for you:

In [ ]:
# For each table: rows, exact duplicate rows, and per column the type, missing values and distinct values
for name, table in [('expeditions', expeditions), ('climbers', climbers)]:
    # Rows and exact duplicates
    print(f'{name}: {len(table)} rows, {table.duplicated().sum()} exact duplicate rows')
    # Type, missing values and distinct values of each column
    display(pd.DataFrame({'type': table.dtypes, 'missing': table.isna().sum(), 'distinct': table.nunique()}))

In [ ]:
# Remove exact duplicate rows
expeditions = expeditions.drop_duplicates()
climbers = climbers.drop_duplicates()

# Rows left
print(len(expeditions), 'expeditions and', len(climbers), 'climbers left')

### Question 1

Prepare both tables for modeling.

**1a.** Which columns will you remove before modeling, and why?

In [ ]:
# Your code here

**Decision**

*Provide your decision here*

**1b.** What will you do with the missing values?

In [ ]:
# Your code here

**Decision**

*Provide your decision here*

**1c. Custom cell.** Any other preparation you decide on (for example, turning text columns into dummy variables).

In [ ]:
# Your code here

**Decision**

*Provide your decision here*

### Question 2

Which climbers and expeditions does this data describe well, and which hardly at all? Figures could help.

**A prompt to start from** (paste it into Gemini, then follow up on what you see):

> *"Make bar charts or histograms of sex, hired staff, season, highest point reached, cause of death, and oxygen sets used. Which groups are barely there, and what does that mean for whom a model will work?"*

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

# **Part 2: Modeling and evaluation** (~20 min)

**How an oxygen plan is scored.** A plan says how many oxygen sets to bring on each expedition, and it can go wrong in two ways:

- **Too few sets:** a climber goes without oxygen high on the mountain.
- **Too many sets:** money is spent, and porters carry weight for nothing.

The agency judges a missing set **10 times** as bad as a spare one. These costs are hypothetical: you can read them as dollars, but they also stand for things that are hard to price, such as a climber's safety or the agency's reputation. The function below adds up the cost of a plan over all expeditions. **Lower is better.**

In [ ]:
# Tool for scoring your oxygen plans (just run this cell)
# Question 3 - the total cost of an oxygen plan over all expeditions:
#   brought: sets the agency brings; actual: sets the climbers needed
#   - brought_cost: cost of each spare set (brought but not needed): money, and weight porters carry for nothing
#   - missing_cost: cost of each missing set (needed but not brought): a climber goes without oxygen high up.
#     The agency judges a missing set 10 times worse than a spare one, so missing_cost=10 and brought_cost=1.
# These costs are hypothetical. You can read them as dollars, but they can also stand for things that are hard
# to price, such as a climber's safety or the agency's reputation. What matters is how they compare (10 to 1).
def oxygen_cost(brought, actual, missing_cost=10, brought_cost=1):
    # Sets missing and sets spare, per expedition
    missing = (actual - brought).clip(lower=0)   # sets that were needed but not brought
    spare = (brought - actual).clip(lower=0)     # sets carried for nothing

    # Total cost over all expeditions
    cost = missing_cost * missing.sum() + brought_cost * spare.sum()

    # Report in plain words, one fact per line
    print('---')
    print(f'Expeditions with not enough oxygen: {(missing > 0).sum():,} of {len(actual):,} ({(missing > 0).mean():.0%})')
    print(f'Sets missing in total: {missing.sum():,.0f}')
    print(f'Spare sets carried but not needed, in total: {spare.sum():,.0f}')
    print(f'Spare sets per expedition, on average: {spare.mean():.1f}')
    print(f'Total cost of this plan: {cost:,.0f}')
    print('---')
    return cost

### Question 3

How many oxygen sets should the agency bring on each 2015–2019 expedition? Compare at most two models, at least one of them easy to interpret. Can you beat a dummy model that brings the same number of sets to every expedition, according to the `oxygen_cost` function above?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

### Question 4

Predict which climbers on 2015–2019 expeditions will die (`died`). Compare at most two models, at least one of them easy to interpret. The agency has decided to **catch as many deaths as possible, without flagging so many climbers that the warning means nothing**. Decide where to draw the line between "high risk" and not, and defend that choice to the agency.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

# **Part 3: Interpretation and drilling deeper** (~20 min)

### Question 5

Which inputs matter most to your best model for each question, and how can you tell?

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

### Question 6

Does your death model work equally well for women and men? Find out why.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*

### Question 7

Does your model hold up year by year after 2014? Explain what you see.

In [ ]:
# Your code here

**Answer**

*Provide your answer here*